In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import pandas as pd
import numpy as np

In [40]:
path = '/workspaces/crmprtd/update_sheet/'
df = pd.read_excel(path + '20250918-Metadata-AllRequiredChanges.xlsx', header = 1)   # pandas automatically uses openpyxl
df_fern = df[df["Network ID"] == 11]

df_fern = df_fern.loc[:, df_fern.notna().any()]
df_fern_vanessa = df_fern[df_fern['Phone number'].str.contains("Vanessa", case=False, na=False)]

fern_v_ted = df_fern_vanessa[['Station_name','native_id', 'lat', 'lon', 'Elevation']].reset_index(drop=True)

fern_v_pcds = df_fern_vanessa[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
fern_v_pcds['Station ID'] = np.int64(fern_v_pcds['Station ID'])

split_cols = fern_v_pcds['Unique Names'].str.split('->', n=1, expand=True)
# Step 2: assign columns
fern_v_pcds['db_station_name'] = split_cols[0].str.strip()

fern_v_pcds[['Station ID', 'Native ID', 'db_station_name']]
fern_v_ted

,Station_name,native_id,lat,lon,Elevation
0,BarrenWx,Barren,54.510444,-126.614341,1051
1,BlackhawkWx,Blackhawk,55.078885,-120.352171,962
2,BoulderWx,Boulder Creek,55.108173,-127.37474,385
3,BowronPit,BowronPit,53.902111,-122.015389,722
4,BulkleyWx,PGTIS1,53.772183,-122.720729,594
5,Canoe Mountain Stn,Canoe Mountain Stn,52.70982,-119.19126,2210
6,Caribou Pass,CaribouPass,58.35718,-129.54637,1744
7,ChapmanWx,Chapman,54.883903,-126.623533,848
8,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720
9,CoalmineWx,CoalmineWx,53.741139,-121.784639,840


### manually revise the modification



In [17]:
fern_v_ted

fern_v_ted.loc[fern_v_ted['Station_name'] == 'Dennis', ['lat', 'lon', 'Elevation']] = [54.76159, -127.46765, 895]
fern_v_ted.loc[fern_v_ted['Station_name'] == 'SmitherCmpd', 'Station_name'] = 'SmithersCmpd'

fern_v_ted_update = fern_v_ted
fern_v_ted_update

,Station_name,native_id,lat,lon,Elevation
0,BarrenWx,Barren,54.510444,-126.614341,1051
1,BlackhawkWx,Blackhawk,55.078885,-120.352171,962
2,BoulderWx,Boulder Creek,55.108173,-127.37474,385
3,BowronPit,BowronPit,53.902111,-122.015389,722
4,BulkleyWx,PGTIS1,53.772183,-122.720729,594
5,Canoe Mountain Stn,Canoe Mountain Stn,52.70982,-119.19126,2210
6,Caribou Pass,CaribouPass,58.35718,-129.54637,1744
7,ChapmanWx,Chapman,54.883903,-126.623533,848
8,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720
9,CoalmineWx,CoalmineWx,53.741139,-121.784639,840


In [26]:
combined_pcds_update = pd.concat([fern_v_pcds[['Station ID', 'db_station_name', 'Native ID']],
                      fern_v_ted_update],
                     axis=1)

combined_pcds_update

combined_pcds_update = combined_pcds_update.rename(columns={
    'Native ID': 'db_native_id',
    'Station ID': 'db_station_id',
    'Station_name': 'updated_station_name',
    'native_id': 'updated_native_id',
    'lat': 'updated_lat',
    'lon': 'updated_lon',
    'Elevation': 'updated_elev'
})
# import os
# print(os.getcwd())

save_path = './comparison_forms/'
combined_pcds_update.to_csv(save_path + '0.0_32station_search_Vanessa_pcds_before_after_summary.csv', index=False)
combined_pcds_update

,db_station_id,db_station_name,db_native_id,updated_station_name,updated_native_id,updated_lat,updated_lon,updated_elev
0,13182,BarrenWx,Barren,BarrenWx,Barren,54.510444,-126.614341,1051
1,12055,Blackhawk,Blackhawk,BlackhawkWx,Blackhawk,55.078885,-120.352171,962
2,12056,Boulder Creek,BoulderCr,BoulderWx,Boulder Creek,55.108173,-127.37474,385
3,12438,Bowron Pit,BowronPit,BowronPit,BowronPit,53.902111,-122.015389,722
4,1604,BulkleyWx,1113694,BulkleyWx,PGTIS1,53.772183,-122.720729,594
5,12057,Canoe Mountain Stn,Canoe,Canoe Mountain Stn,Canoe Mountain Stn,52.70982,-119.19126,2210
6,13220,Caribou Pass,Caribou Pass,Caribou Pass,CaribouPass,58.35718,-129.54637,1744
7,13184,ChapmanWx,Chapman,ChapmanWx,Chapman,54.883903,-126.623533,848
8,13070,Chief Lake,Chief Lake,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720
9,12439,Coalmine Wx,CoalmineWx,CoalmineWx,CoalmineWx,53.741139,-121.784639,840


### Connect to db and update metadata

In [25]:
import sqlalchemy as sa
from sqlalchemy.orm import Session


engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)



for station_name, station_id in zip(combined_pcds_update['db_station_name'], combined_pcds_update['db_station_id']):
    query = sa.text("""
        SELECT DISTINCT m.station_name, m.lat, m.lon, m.elev, s.native_id, s.mod_user
        FROM meta_history AS m
        JOIN meta_station AS s ON m.station_id = s.station_id
        WHERE s.network_id = 11
          AND m.station_name = :station_name
          And m.station_id = :station_id

    """)

    with engine.begin() as conn:  # begin() ensures commit/rollback
        result = conn.execute(query, {"station_name": station_name, "station_id": station_id})
        row = result.first()  # get the first row, None if not found

    if row:
        print(f"Yes, Station '{station_name}' exists in the database.")
    else:
        print(f"No, Station '{station_name}' does NOT exist in the database.")

Yes, Station 'BarrenWx' exists in the database.
Yes, Station 'Blackhawk' exists in the database.
Yes, Station 'Boulder Creek' exists in the database.
Yes, Station 'Bowron Pit' exists in the database.
Yes, Station 'BulkleyWx' exists in the database.
Yes, Station 'Canoe Mountain Stn' exists in the database.
Yes, Station 'Caribou Pass' exists in the database.
Yes, Station 'ChapmanWx' exists in the database.
Yes, Station 'Chief Lake' exists in the database.
Yes, Station 'Coalmine Wx' exists in the database.
Yes, Station 'CPFWx' exists in the database.
Yes, Station 'Crystal Lake' exists in the database.
Yes, Station 'Dennis' exists in the database.
Yes, Station 'Dunster' exists in the database.
Yes, Station 'EndakoWx' exists in the database.
Yes, Station 'GeorgeWx' exists in the database.
Yes, Station 'Gnat Pass' exists in the database.
Yes, Station 'Gunnel3' exists in the database.
Yes, Station 'GunnelWx' exists in the database.
Yes, Station 'Hourglass' exists in the database.
Yes, Station

In [35]:
### Information before update
# import pandas as pd
# import sqlalchemy as sa
# from sqlalchemy.orm import Session

# engine = sa.create_engine(
#     "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca"
#     "&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass",
#     echo=False
# )

# session = Session(engine)

# results_list = []   # store each row as dict

# query = sa.text("""
#     SELECT DISTINCT m.station_id, m.station_name, m.lat, m.lon, m.elev, 
#                     s.native_id, s.mod_user
#     FROM meta_history AS m
#     JOIN meta_station AS s ON m.station_id = s.station_id
#     WHERE s.network_id = 11
#       AND m.station_name = :station_name
#       AND m.station_id = :station_id
# """)

# for station_name, station_id in zip(combined_pcds_update['db_station_name'], combined_pcds_update['db_station_id']):
    
#     with engine.begin() as conn:
#         result = conn.execute(query, {"station_name": station_name, "station_id": station_id})
#         row = result.first()

#     if row:
#         # SQLAlchemy Row → dict
#         results_list.append(dict(row._mapping))
#     else:
#         results_list.append({
#             "station_name": station_name,
#             "lat": None,
#             "lon": None,
#             "elev": None,
#             "native_id": None,
#             "mod_user": None
#         })

# # Convert to DataFrame
# df_results = pd.DataFrame(results_list)

# df_results.to_csv(save_path + '0.0_32station_search_Vanessa_pcds_info_before_update.csv', index=False)
# df_results

,station_id,station_name,lat,lon,elev,native_id,mod_user
0,13182,BarrenWx,54.51044,-126.6143,1051,Barren,crmp
1,12055,Blackhawk,55.0789,-120.352,962.492,Blackhawk,crmp
2,12056,Boulder Creek,55.1082,-127.375,385.101,BoulderCr,crmp
3,12438,Bowron Pit,53.902111,-122.015389,722,BowronPit,crmp
4,1604,BulkleyWx,53.771956,-122.720939,599,1113694,crmp
5,12057,Canoe Mountain Stn,52.7098,-119.191,2210,Canoe,crmp
6,13220,Caribou Pass,58.35718,-129.54637,1744,Caribou Pass,crmp
7,13184,ChapmanWx,54.8839,-126.6235,848,Chapman,crmp
8,13070,Chief Lake,54.23821,-122.8775,726,Chief Lake,crmp
9,12439,Coalmine Wx,53.741139,-121.784639,840,CoalmineWx,crmp


### Update

In [36]:
update_meta_history = sa.text("""
    UPDATE meta_history
    SET station_name = :new_name,
        lat = :new_lat,
        lon = :new_lon,
        elev = :new_elev
    WHERE station_id = :station_id
""")

update_meta_station = sa.text("""
    UPDATE meta_station
    SET native_id = :new_native_id
    WHERE station_id = :station_id
""")

with engine.begin() as conn:

    for idx, row in combined_pcds_update.iterrows():

        conn.execute(update_meta_history, {
            "new_name": row["updated_station_name"],
            "new_lat": row["updated_lat"],
            "new_lon": row["updated_lon"],
            "new_elev": row["updated_elev"],
            "station_id": row["db_station_id"]
        })

        conn.execute(update_meta_station, {
            "new_native_id": row["updated_native_id"],
            "station_id": row["db_station_id"]
        })

### Check

In [41]:
query_check = sa.text("""
    SELECT m.station_id, m.station_name, m.lat, m.lon, m.elev,
           s.native_id, s.mod_user
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE m.station_id = ANY(:station_ids)
""")

station_ids = combined_pcds_update['db_station_id'].tolist()

with engine.begin() as conn:
    result = conn.execute(query_check, {"station_ids": station_ids})
    updated_rows = result.fetchall()

# Convert to pandas DataFrame for easy comparison
df_updated = pd.DataFrame([dict(row._mapping) for row in updated_rows])

df_updated.to_csv(save_path + '0.0_32station_search_Vanessa_pcds_info_after_update.csv', index=False)
df_results

,station_id,station_name,lat,lon,elev,native_id,mod_user
0,13182,BarrenWx,54.51044,-126.6143,1051,Barren,crmp
1,12055,Blackhawk,55.0789,-120.352,962.492,Blackhawk,crmp
2,12056,Boulder Creek,55.1082,-127.375,385.101,BoulderCr,crmp
3,12438,Bowron Pit,53.902111,-122.015389,722,BowronPit,crmp
4,1604,BulkleyWx,53.771956,-122.720939,599,1113694,crmp
5,12057,Canoe Mountain Stn,52.7098,-119.191,2210,Canoe,crmp
6,13220,Caribou Pass,58.35718,-129.54637,1744,Caribou Pass,crmp
7,13184,ChapmanWx,54.8839,-126.6235,848,Chapman,crmp
8,13070,Chief Lake,54.23821,-122.8775,726,Chief Lake,crmp
9,12439,Coalmine Wx,53.741139,-121.784639,840,CoalmineWx,crmp


### Some comparison

In [ ]:
df = pd.read_excel(path + '2025118-MeteorologicalNetworks.xlsx', header = 0)   # pandas automatically uses openpyxl
df_fern = df[df['Contact_Name'].str.contains("Vanessa", case=False, na=False)]
df_fern_vane = df_fern[['lat', 'lon', 'elev', 'Station_name', 'native_id']]
df_fern_vane

In [54]:

# 1. Standardize column names first (optional but recommended)
fern = fern_v_ted.rename(columns={'Elevation': 'elev'})
vane = df_fern_vane.rename(columns={'Elevation': 'elev'})

# 2. Merge on station_name
merged = fern.merge(
    vane,
    on='Station_name',
    how='inner',
    suffixes=('_ted', '_vane')
)

# 3. Compare key columns
merged['lat_match']  = merged['lat_ted']   == merged['lat_vane']
merged['lon_match']  = merged['lon_ted']   == merged['lon_vane']
merged['elev_match'] = merged['elev_ted']  == merged['elev_vane']
merged['id_match']   = merged['native_id_ted'] == merged['native_id_vane']

# 4. Whether all match
merged['all_match'] = (
    merged['lat_match'] &
    merged['lon_match'] &
    merged['elev_match'] &
    merged['id_match']
)

# 5. Show results
merged[['Station_name', 'lat_match', 'lon_match', 'elev_match', 'id_match', 'all_match']]

,Station_name,lat_match,lon_match,elev_match,id_match,all_match
0,BarrenWx,True,True,True,True,True
1,BlackhawkWx,True,True,True,True,True
2,BoulderWx,True,True,True,True,True
3,BowronPit,True,True,True,True,True
4,BulkleyWx,True,True,True,True,True
5,Canoe Mountain Stn,True,True,True,True,True
6,Caribou Pass,True,True,True,True,True
7,ChapmanWx,True,True,True,True,True
8,ChiefLakeWx,True,True,True,True,True
9,CoalmineWx,True,True,True,True,True


### Comparison of Ted's sheet with the change part, rn ca neglect the yellow change part

In [66]:
# Make sure the station name in Ted's sheet consistent with the change part

# Step 1: split on '->'
split_cols = fern_v_pcds['Unique Names'].str.split('->', n=1, expand=True)
# Step 2: assign columns
fern_v_pcds['db_station_name'] = split_cols[0].str.strip()
# Step 3: if no '->', use old name also as new name
fern_v_pcds['new_station_name'] = split_cols[1].str.strip().fillna(split_cols[0].str.strip())

station_name_comparison = fern_v_pcds['new_station_name'] == fern_v_ted['Station_name'] ## Seems 28 is wrong, should be SmithersCmpd rather than SmitherCmpd


station_name_summary = pd.DataFrame({
    'station_name_pcds_re': fern_v_pcds['new_station_name'],
    'station_name_ted': fern_v_ted['Station_name'],
    'station_name_match': station_name_comparison,
})

station_name_summary    # display only rows with any mismatc

,station_name_pcds_re,station_name_ted,station_name_match
0,BarrenWx,BarrenWx,True
1,BlackhawkWx,BlackhawkWx,True
2,BoulderWx,BoulderWx,True
3,BowronPit,BowronPit,True
4,BulkleyWx,BulkleyWx,True
5,Canoe Mountain Stn,Canoe Mountain Stn,True
6,Caribou Pass,Caribou Pass,True
7,ChapmanWx,ChapmanWx,True
8,ChiefLakeWx,ChiefLakeWx,True
9,CoalmineWx,CoalmineWx,True


In [67]:
# Make sure the LAT/LON/ELEV in Ted's sheet consistent with the change part


split_cols_geo = fern_v_pcds['Unique Location (String)'].str.split('->', n=1, expand=True)
fern_v_pcds['new_lat_lon_elev'] = split_cols_geo[1].str.strip().fillna(split_cols_geo[0].str.strip())

import re

# Extract lon, lat, elevation
def parse_lat_lon_elev(text):
    # regex to capture: lon value, W/E, lat value, N/S, elevation
    pattern = r'([\d\.]+)\s*([WE]),\s*([\d\.]+)\s*([NS]),\s*Elev\.\s*([\d\.]+)'
    m = re.search(pattern, text)
    
    if not m:
        return None, None, None

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    # Convert numbers
    lon = float(lon_val) * (-1 if lon_dir == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir == "S" else 1)
    elev = float(elev_val)

    return lat, lon, elev

# Apply to dataframe
fern_v_pcds[['lat', 'lon', 'elev']] = fern_v_pcds['new_lat_lon_elev'].apply(
    lambda x: pd.Series(parse_lat_lon_elev(x))
)

# Force elevation to be integer
fern_v_pcds['elev'] = fern_v_pcds['elev'].astype('Int64')




geo_summary = pd.DataFrame({
    'station_name_ted': fern_v_ted['Station_name'],
    
    'lat_pcds_re': fern_v_pcds['lat'],
    'lat_ted': fern_v_ted['lat'],
    'lat_match': fern_v_pcds['lat'] == fern_v_ted['lat'],

    'lon_pcds_re': fern_v_pcds['lon'],
    'lon_ted': fern_v_ted['lon'],
    'lon_match': fern_v_pcds['lon'] == fern_v_ted['lon'],

    'elev_pcds_re': fern_v_pcds['elev'],
    'elev_ted': fern_v_ted['Elevation'],
    'elev_match': fern_v_pcds['elev'] == fern_v_ted['Elevation']
})

geo_summary    # display only rows with any mismatch

,station_name_ted,lat_pcds_re,lat_ted,lat_match,lon_pcds_re,lon_ted,lon_match,elev_pcds_re,elev_ted,elev_match
0,BarrenWx,54.510440,54.510444,False,-126.614300,-126.614341,False,1051,1051,True
1,BlackhawkWx,55.078900,55.078885,False,-120.352200,-120.352171,False,962,962,True
2,BoulderWx,55.108200,55.108173,False,-127.374700,-127.37474,False,385,385,True
3,BowronPit,53.902111,53.902111,True,-122.015389,-122.015389,True,722,722,True
4,BulkleyWx,53.772200,53.772183,False,-122.720700,-122.720729,False,594,594,True
5,Canoe Mountain Stn,52.709800,52.70982,False,-119.191000,-119.19126,False,2210,2210,True
6,Caribou Pass,58.357180,58.35718,True,-129.546370,-129.54637,True,1744,1744,True
7,ChapmanWx,54.883900,54.883903,False,-126.623500,-126.623533,False,848,848,True
8,ChiefLakeWx,54.238000,54.238028,False,-122.877300,-122.877306,False,720,720,True
9,CoalmineWx,53.741139,53.741139,True,-121.784639,-121.784639,True,840,840,True


In [68]:
fern_v_pcds['db_station_name']

0               BarrenWx
1              Blackhawk
2          Boulder Creek
3             Bowron Pit
4              BulkleyWx
5     Canoe Mountain Stn
6           Caribou Pass
7              ChapmanWx
8             Chief Lake
9            Coalmine Wx
10                 CPFWx
11          Crystal Lake
12                Dennis
13               Dunster
14              EndakoWx
15              GeorgeWx
16             Gnat Pass
17               Gunnel3
18              GunnelWx
19             Hourglass
20       Hudson Bay Mtn2
21               Kluskus
22         Mackenzie Jxn
23          MiddleforkWx
24               NondaWx
25      Perkins Peak Stn
26                PinkWx
27              SaxtonWx
28          SmithersCmpd
29               Sunbeam
30            ThompsonWx
31       Willow-BowronWx
Name: db_station_name, dtype: object

In [69]:
fern_v_ted['db_station_name'] = fern_v_pcds['db_station_name']
fern_v_ted

,Station_name,native_id,lat,lon,Elevation,db_station_name
0,BarrenWx,Barren,54.510444,-126.614341,1051,BarrenWx
1,BlackhawkWx,Blackhawk,55.078885,-120.352171,962,Blackhawk
2,BoulderWx,Boulder Creek,55.108173,-127.37474,385,Boulder Creek
3,BowronPit,BowronPit,53.902111,-122.015389,722,Bowron Pit
4,BulkleyWx,PGTIS1,53.772183,-122.720729,594,BulkleyWx
5,Canoe Mountain Stn,Canoe Mountain Stn,52.70982,-119.19126,2210,Canoe Mountain Stn
6,Caribou Pass,CaribouPass,58.35718,-129.54637,1744,Caribou Pass
7,ChapmanWx,Chapman,54.883903,-126.623533,848,ChapmanWx
8,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720,Chief Lake
9,CoalmineWx,CoalmineWx,53.741139,-121.784639,840,Coalmine Wx


In [71]:
fern_v_ted

,Station_name,native_id,lat,lon,Elevation,db_station_name
0,BarrenWx,Barren,54.510444,-126.614341,1051,BarrenWx
1,BlackhawkWx,Blackhawk,55.078885,-120.352171,962,Blackhawk
2,BoulderWx,Boulder Creek,55.108173,-127.37474,385,Boulder Creek
3,BowronPit,BowronPit,53.902111,-122.015389,722,Bowron Pit
4,BulkleyWx,PGTIS1,53.772183,-122.720729,594,BulkleyWx
5,Canoe Mountain Stn,Canoe Mountain Stn,52.70982,-119.19126,2210,Canoe Mountain Stn
6,Caribou Pass,CaribouPass,58.35718,-129.54637,1744,Caribou Pass
7,ChapmanWx,Chapman,54.883903,-126.623533,848,ChapmanWx
8,ChiefLakeWx,ChiefWx,54.238028,-122.877306,720,Chief Lake
9,CoalmineWx,CoalmineWx,53.741139,-121.784639,840,Coalmine Wx


In [72]:
1.24/1.5

0.8266666666666667

In [73]:
import pandas as pd
import sqlalchemy as sa

for station_name, station_id, lat, lon, elev, native_id in zip(
        fern_v_ted['Station_name'],
        fern_v_pcds['Station ID'],
        fern_v_ted['lat'],
        fern_v_ted['lon'],
        fern_v_ted['Elevation'],
        fern_v_ted['native_id']):
    
    select_query = sa.text("""
        SELECT m.station_name, m.lat, m.lon, m.elev, s.native_id
        FROM meta_history AS m
        JOIN meta_station AS s ON m.station_id = s.station_id
        WHERE s.network_id = 11
          AND m.station_id = :station_id
    """)

    with engine.begin() as conn:
        row_before = conn.execute(select_query, {"station_id": int(station_id)}).first()
    
    if row_before:
        print(f"Before update for station_id {station_id}: {row_before}")
    else:
        print(f"No row found for station_id {station_id}")

Before update for station_id 13182: ('BarrenWx', Decimal('54.51044'), Decimal('-126.6143'), Decimal('1051'), 'Barren')
Before update for station_id 12055: ('Blackhawk', Decimal('55.0789'), Decimal('-120.352'), Decimal('962.492'), 'Blackhawk')
Before update for station_id 12056: ('Boulder Creek', Decimal('55.1082'), Decimal('-127.375'), Decimal('385.101'), 'BoulderCr')
Before update for station_id 12438: ('Bowron Pit', Decimal('53.902111'), Decimal('-122.015389'), Decimal('722'), 'BowronPit')
Before update for station_id 1604: ('BulkleyWx', Decimal('53.771956'), Decimal('-122.720939'), Decimal('599'), '1113694')
Before update for station_id 12057: ('Canoe Mountain Stn', Decimal('52.7098'), Decimal('-119.191'), Decimal('2210'), 'Canoe')
Before update for station_id 13220: ('Caribou Pass', Decimal('58.35718'), Decimal('-129.54637'), Decimal('1744'), 'Caribou Pass')
Before update for station_id 13184: ('ChapmanWx', Decimal('54.8839'), Decimal('-126.6235'), Decimal('848'), 'Chapman')
Before

In [74]:


# for station_name, station_id, lat, lon, elev, native_id in zip(
#         fern_v_ted['Station_name'],
#         fern_v_pcds['Station ID'],
#         fern_v_ted['lat'],
#         fern_v_ted['lon'],
#         fern_v_ted['Elevation'],
#         fern_v_ted['native_id']):
    
#     update_query = sa.text("""
#         UPDATE meta_history AS m
#         SET station_name = :station_name,
#             lat = :lat,
#             lon = :lon,
#             elev = :elev,
#             native_id = :native_id
#         FROM meta_station AS s
#         WHERE s.network_id = 11
#           AND m.station_id = :station_id
#     """)

#     # Begin a transaction
#     with engine.begin() as conn:
#         conn.execute(update_query, {
#             "station_name": station_name,
#             "lat": lat,
#             "lon": lon,
#             "elev": elev,
#             "native_id": native_id,
#             "station_id": int(station_id)  # ensure plain int
#         })

#     print(f"Updated station_id {station_id} -> {station_name}")